# C7-cnn-transfer — Practice p27

**Type:** scenario analysis · **Difficulty:** advanced · **Concepts:** layer-freezing, requires-grad, cnn-training

**Time budget:** 75 minutes

## Part I — commit independent-control predictions (8 points)

A convolution–BatchNorm–ReLU trunk starts in training mode, with trainable
parameters and ordinary gradient tracking. Three cases each change exactly one
control from an identical fresh baseline:

1. `eval_case`: call `trunk.eval()`; parameters remain trainable and gradient
   tracking remains enabled.
2. `frozen_case`: keep `trunk.train()` but set every parameter's
   `requires_grad=False`; ordinary gradient tracking remains enabled.
3. `inference_case`: keep training mode and trainable parameters, but run the
   forward inside `torch.inference_mode()`.

For each, predict
`(output_has_gradient_graph, output_changes_from_baseline,
running_buffers_change, output_is_inference_tensor)`.
Assign exact bool tuples `prediction_eval`, `prediction_frozen`, and
`prediction_inference` **before** the marked verifier, then explain in 3–5
sentences why mode, trainability, and graph context are independent.

## Part II — selective training and evaluation audit (12 points)

Only after Part I's marked verifier agrees, use the supplied seeded template,
data, and exact order:

1. make `train_model = deepcopy(template)`;
2. freeze only parameters whose names start with `conv` or `bn`;
3. snapshot all parameters and buffers;
4. construct SGD (`lr=0.18`) from only trainable parameters;
5. call `train()` and run exactly 18 full-batch CE steps;
6. audit mode, flags, exact optimizer ownership, final gradient names,
   bitwise frozen immobility, allowed parameter movement, finite loss reduction
   to at most `0.80 * initial`, and training-mode BN-buffer movement;
7. snapshot the trained state, call `eval()`, and under `no_grad()` perform two
   same-input evaluations; audit no graph, identical logits at
   `atol=1e-10, rtol=1e-8`, and bitwise unchanged parameters and buffers.

Assign the combined boolean `training_evaluation_certificate`.

**Banned — zero points for Part I:** `backward()`, optimizers, changing the
supplied trunk/input, running or reading the verifier before all predictions
are committed, moving predictions below the marked verifier, or assigning any
observed/verdict value yourself.

**Banned — zero points for Part II:** pretrained weights, downloads/network,
changing the supplied template/data/seed/order/steps/hyperparameters, evaluating
in training mode, including frozen parameters in the optimizer, editing `.grad`
or parameter `.data`, or reporting only loss without separate mode/flag/graph/
ownership/parameter/buffer audits.

In [ ]:
prediction_eval: tuple[bool, bool, bool, bool] = ...
prediction_frozen: tuple[bool, bool, bool, bool] = ...
prediction_inference: tuple[bool, bool, bool, bool] = ...

committed_predictions = [prediction_eval, prediction_frozen, prediction_inference]


**Your 3–5 sentence explanation:**



### MARKED VERIFICATION CELL

This cell fails closed and prints only one agreement bit; it never reveals per-case observations.

In [ ]:
from copy import deepcopy

import torch
import torch.nn as nn

if len(committed_predictions) != 3 or any(
    value is Ellipsis
    or not isinstance(value, tuple)
    or len(value) != 4
    or any(type(flag) is not bool for flag in value)
    for value in committed_predictions
):
    raise RuntimeError("commit exactly three four-bool prediction tuples first")

SEED = 20260804
torch.manual_seed(SEED)
base_trunk = nn.Sequential(
    nn.Conv2d(3, 5, kernel_size=3, padding=1, bias=False),
    nn.BatchNorm2d(5),
    nn.ReLU(),
)
x = torch.randn(4, 3, 7, 7)

def run_case(*, training, parameters_require_grad, inference):
    trunk = deepcopy(base_trunk)
    trunk.train(training)
    for parameter in trunk.parameters():
        parameter.requires_grad_(parameters_require_grad)
    bn = trunk[1]
    before_mean = bn.running_mean.clone()
    before_var = bn.running_var.clone()
    context = torch.inference_mode() if inference else torch.enable_grad()
    with context:
        output = trunk(x)
    buffers_change = not (
        torch.allclose(bn.running_mean, before_mean, atol=1e-7, rtol=0)
        and torch.allclose(bn.running_var, before_var, atol=1e-7, rtol=0)
    )
    return (
        bool(output.requires_grad), output.detach().clone(), buffers_change,
        bool(torch.is_inference(output)),
    )

_, baseline_output, _, _ = run_case(
    training=True, parameters_require_grad=True, inference=False
)
settings = [
    dict(training=False, parameters_require_grad=True, inference=False),
    dict(training=True, parameters_require_grad=False, inference=False),
    dict(training=True, parameters_require_grad=True, inference=True),
]
observed = []
for case in settings:
    graph, output, buffers, is_inference = run_case(**case)
    changed = not torch.allclose(output, baseline_output, atol=1e-7, rtol=0)
    observed.append((graph, changed, buffers, is_inference))
all_predictions_agree = all(
    hand == seen for hand, seen in zip(committed_predictions, observed)
)
print("all_agree:", all_predictions_agree)
del observed, baseline_output


In [ ]:
if all_predictions_agree is not True:
    raise RuntimeError("Part I must be committed and correct before Part II")

torch.set_default_dtype(torch.float64)
torch.manual_seed(SEED)

class TrainingAuditCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 5, 3, padding=1)
        self.bn = nn.BatchNorm2d(5)
        self.dropout = nn.Dropout(p=0.15)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Linear(5, 3)

    def forward(self, value):
        value = torch.relu(self.bn(self.conv(value)))
        value = self.dropout(value)
        return self.head(torch.flatten(self.pool(value), 1))

template = TrainingAuditCNN()
generator = torch.Generator(device="cpu").manual_seed(SEED)
train_X = 0.04 * torch.randn(18, 1, 7, 7, generator=generator)
train_y = torch.arange(18, dtype=torch.long) % 3
train_X[train_y == 0, :, :, 1:3] += 1.0
train_X[train_y == 1, :, 4:6, :] += 1.0
diagonal = torch.arange(7)
class_two = train_X[train_y == 2].clone()
class_two[:, :, diagonal, diagonal] += 1.0
train_X[train_y == 2] = class_two
train_X[train_y == 0] -= 0.8
train_X[train_y == 2] += 0.8
criterion = nn.CrossEntropyLoss()


In [ ]:
# Follow the seven-step Part II order exactly.
# YOUR CODE HERE
train_model = ...
parameter_before = ...
buffer_before = ...
optimizer = ...
loss_history = ...

train_mode_audit = ...
trainability_audit = ...
optimizer_ownership_audit = ...
gradient_names = ...
frozen_parameters_unchanged = ...
allowed_parameter_movement = ...
training_buffers_moved = ...
loss_audit = ...

evaluation_mode_audit = ...
evaluation_has_no_graph = ...
evaluation_logits_repeat = ...
evaluation_parameters_unchanged = ...
evaluation_buffers_unchanged = ...

training_evaluation_certificate = ...
